# Beat2Vibe — reproducible notebook workflow
Run in the repository root. **Audio and original human-annotated pair files are not included.** Numerical results are generated only by executing models on the supplied real dataset.

In [ ]:
from pathlib import Path
import yaml
config = yaml.safe_load(Path("config.yaml").read_text())
config

## 1. Prepare audio, features and track-disjoint splits
Provide `data/tracks.csv` and `data/pairs.csv` as documented in `data/README.md`.

In [ ]:
!python run.py prepare --tracks data/tracks.csv --pairs data/pairs.csv --out data/processed

## 2. Self-supervised acoustic/vibe learning and train-only prototypes

In [ ]:
!python run.py pretrain --data data/processed --out results/vibe_pretrained.pt

## 3. Train supervised classification and ranking networks

In [ ]:
!python run.py train --data data/processed --ssl results/vibe_pretrained.pt --variant full --out results/full.pt
!python run.py train --data data/processed --ssl results/vibe_pretrained.pt --variant fixed --out results/fixed.pt
!python run.py train --data data/processed --variant baseline --out results/baseline.pt

## 4. Evaluate on held-out test tracks
Ranking uses unjudged sampled alternatives; see the README for limitations. The generated metrics do not automatically validate the numerical values from the manuscript.

In [ ]:
!python run.py evaluate --data data/processed --checkpoint results/full.pt --out results/full_test
!python run.py evaluate --data data/processed --checkpoint results/fixed.pt --out results/fixed_test
!python run.py evaluate --data data/processed --checkpoint results/baseline.pt --out results/baseline_test

## 5. Verify significance and inspect adaptive cue weights

In [ ]:
!python run.py compare --a results/full_test_ranking.csv --b results/fixed_test_ranking.csv --out results/paired_comparison.json
!python run.py plot-weights --ranking-csv results/full_test_ranking.csv --out results/controller_weights.png